# Rarest signatures: peak detection

Step 1 of 3. Finds when and where isolated rarest signatures cluster and labels the top peaks.

Input: `final_rarest_4_nbr_count.parquet` (rarest observations with neighbour counts from `examples/density_estimation.py`).
Output: `final_rarest_4_nbr_count_0_m_3_peaks.parquet`, read by `rarest_peaks_2_time_series.ipynb`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime
import logging
import pathlib

import numpy as np
import polars as pl
from gpm.utils.geospatial import Extent

from pmw_analysis.analysis.spatial_visualization import plot_variables_on_map
from pmw_analysis.constants import COLUMN_LON, COLUMN_LAT, DIR_IMAGES, ArgSurfaceType, COLUMN_TIME, COLUMN_COUNT, \
    FILE_DF_FINAL_RAREST
from pmw_analysis.constants import DIR_PMW_ANALYSIS, TC_COLUMNS, ArgTransform
from pmw_analysis.processing.filter import filter_by_signature_occurrences_count, filter_by_value_range, \
    get_filter_expr_from_value_range
from pmw_analysis.preprocessing.transforms import get_transformation_function
from pmw_analysis.utils.io import combine_paths, file_to_dir
from pmw_analysis.utils.logging import timing

from examples.signatures_evolution import _detect_peaks, take_k_peaks_by_zones

## Settings

In [ ]:
arg_transform = ArgTransform.V4
arg_surface_type = ArgSurfaceType.LAND
df_dir_path = pathlib.Path(DIR_PMW_ANALYSIS) / arg_transform.value / arg_surface_type.value
max_count = 4  # max count of the rarest signatures, as in examples/density_estimation.py

transform = get_transformation_function(arg_transform)
quant_columns = transform(TC_COLUMNS)

logging.basicConfig(level=logging.INFO)

## Isolated rarest signatures

`df_m`: signatures seen at least `m_occurrences` times; `*_nbr_0`: observations with no neighbours (`nbr_count_1 == 0`).

In [ ]:
df_path = df_dir_path / f"{FILE_DF_FINAL_RAREST[:FILE_DF_FINAL_RAREST.find('.')]}_{max_count}_nbr_count.parquet"

df = pl.read_parquet(df_path)
images_dir = combine_paths(path_base=pathlib.Path("..") / DIR_IMAGES, path_rel=file_to_dir(df_path), path_rel_base=DIR_PMW_ANALYSIS)
images_dir.mkdir(parents=True, exist_ok=True)

df = df.with_columns(pl.lit(1).alias(COLUMN_COUNT))
with timing("Calculating"):
    # Only consider signatures that appeared at least a specified number of times
    m_occurrences = 3
    df_m = filter_by_signature_occurrences_count(df, m_occurrences, quant_columns)

    df_nbr_0 = filter_by_value_range(df, "nbr_count_1", (0, 1))

    df_m_nbr_0 = filter_by_value_range(df_m, "nbr_count_1", (0, 1))

    columns_to_plot = [COLUMN_LON, COLUMN_LAT, COLUMN_COUNT]

    plot_variables_on_map(df[columns_to_plot], arg_transform, images_dir)
    plot_variables_on_map(df_m[columns_to_plot], arg_transform, images_dir, title_text_suffix=f" (m = {m_occurrences})", file_name_suffix=f"_m_{m_occurrences}")
    plot_variables_on_map(df_nbr_0[columns_to_plot], arg_transform, images_dir, title_text_suffix=" (nbr = 0)", file_name_suffix="_nbr_0")
    plot_variables_on_map(df_m_nbr_0[columns_to_plot], arg_transform, images_dir, title_text_suffix=f" (m = {m_occurrences}, nbr = 0)", file_name_suffix=f"_m_{m_occurrences}_nbr_0")

## Peaks in time (1-day bins) and in lon/lat (1° bins)

In [ ]:
bin_sizes_time = {COLUMN_TIME: datetime.timedelta(days=1)}
bin_ranges_time = {}

peaks_time = _detect_peaks(df_m_nbr_0[[COLUMN_TIME]], bin_sizes_time, bin_ranges_time, images_dir)

bin_sizes_lon_lat = {COLUMN_LON: 1, COLUMN_LAT: 1}
bin_ranges_lon_lat = {COLUMN_LON: (-180, 180), COLUMN_LAT: (-75, 75)}

peaks_lon_lat = _detect_peaks(df_m_nbr_0[[COLUMN_LON, COLUMN_LAT]], bin_sizes_lon_lat, bin_ranges_lon_lat, images_dir)

## Zones

0: rest of the world, 1: Greenland box (60°W–0°, 60°N–90°N), 2: south of 60°S.

In [ ]:
zones = pl.Series(np.zeros(len(df_m_nbr_0), dtype=int))
extent_greenland = Extent(xmin=-60, xmax=0, ymin=60, ymax=90)  # gpm.utils.geospatial.get_country_extent("Greenland")
extent_antarctic = Extent(xmin=-180, xmax=180, ymin=-90, ymax=-60)
for i, extent in enumerate([extent_greenland, extent_antarctic]):
    filter_expr = get_filter_expr_from_value_range([COLUMN_LON, COLUMN_LAT], np.array(extent).reshape((2, 2)))
    zones = df_m_nbr_0.select(pl.when(filter_expr).then(pl.lit(i + 1)).otherwise(zones)).to_series()

In [ ]:
columns_to_plot = [COLUMN_LON, COLUMN_LAT, "zones"]
plot_variables_on_map(df_m_nbr_0.with_columns(zones.alias("zones"))[columns_to_plot], arg_transform, images_dir,
                      title_text_suffix=f" (m = {m_occurrences}, nbr = 0)",
                      file_name_suffix=f"_m_{m_occurrences}_nbr_0")

## Top-k peaks per zone

In [ ]:
k = {0: 12, 1: 4, 2: 4}
peaks_time_10 = take_k_peaks_by_zones(peaks_time, k, zones)
peaks_lon_lat_10 = take_k_peaks_by_zones(peaks_lon_lat, k, zones)

df_m_nbr_0 = df_m_nbr_0.with_columns(peaks_time_10.to_series().alias("peaks_time"))
df_m_nbr_0 = df_m_nbr_0.with_columns(peaks_lon_lat_10.to_series().alias("peaks_lon_lat"))

columns_to_plot = [COLUMN_LON, COLUMN_LAT, COLUMN_COUNT]
columns_to_plot += ["peaks_time", "peaks_lon_lat"]
with timing("Plotting on map"):
    m_occurrences_text = "" if m_occurrences == 1 else f"; Signature occurred at least {m_occurrences} times."
    plot_variables_on_map(df_m_nbr_0[columns_to_plot], arg_transform,
                          images_dir=images_dir,
                          title_text_suffix=m_occurrences_text,
                          file_name_suffix=f"_m_{m_occurrences}_nbr_0_peaks",
                          extent=None,
                          threshold_cat=20)
# df_m.write_parquet(df_path.parent / f"{df_path.stem}_peaks.parquet")

## Save

In [ ]:
df_m_nbr_0.write_parquet(df_dir_path / f"{df_path.stem}_0_m_{m_occurrences}_peaks.parquet")